In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.46.3

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

### Load Dataset

In [ ]:
train_df = pd.read_csv('nlp_train.csv')
test_df  = pd.read_csv('nlp_test.csv')

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(train_df.head(3))

### Label Encoding

In [ ]:
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['risk'])
test_df['label']  = le.transform(test_df['risk'])

print('Classes:', le.classes_)

### Tokenization Setup

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(texts, max_length=64):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='tf'
    )

train_enc = tokenize(train_df['text'].tolist())
test_enc  = tokenize(test_df['text'].tolist())

### Encode Dataset

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_enc),
    train_df['label'].values
)).shuffle(1000).batch(32)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_enc),
    test_df['label'].values
)).batch(32)

print(f'Train batches: {len(train_dataset)}')
print(f'Test batches : {len(test_dataset)}')

### Class Weights & Load Model & Compile Model

In [ ]:
classes = np.unique(train_df['label'])
weights = compute_class_weight('balanced', classes=classes, y=train_df['label'])
class_weights = dict(zip(classes, weights))
print('Class weights:', class_weights)

model = TFBertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3
)

optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5)
loss      = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric    = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')

model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

### GPU check

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

In [ ]:
!nvidia-smi

### Train Model

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='accuracy',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

### Evaluation

In [ ]:
all_preds, all_labels = [], []

for batch, labels in test_dataset:
    outputs = model(batch, training=False)
    preds   = tf.argmax(outputs.logits, axis=1).numpy()
    all_preds.extend(preds)
    all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=le.classes_))

### Prediction Function

In [ ]:
def predict(text):
    encoded = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=64,
        return_tensors='tf'
    )
    outputs = model(encoded, training=False)
    pred    = tf.argmax(outputs.logits, axis=1).numpy()[0]
    return le.inverse_transform([pred])[0]

In [ ]:
text = "Dermatology case: 45 year old male patient. Lesion location: back. Diagnosis method: histo."
print('Risk:', predict(text))

### Save Model

In [ ]:
import pickle

model.save_pretrained('nlp_model')
tokenizer.save_pretrained('nlp_model')

with open('nlp_model/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('Saved!')

In [ ]:
!zip -r nlp_model.zip nlp_model

In [ ]:
from google.colab import files

files.download('nlp_model.zip')